In [ ]:
from rpy2.robjects import r, numpy2ri, pandas2ri, globalenv
from rpy2.robjects.vectors import StrVector
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata, scvelo as scv


# ═══════════════ 1. READ NEG-CONTROL CSVs ══════════════════════════════════
data_dir, dims = "./data/neg_control", [2, 8, 32, 128]
datasets = {
    f"d{d}": {
        "X": pd.read_csv(os.path.join(data_dir, f"X_d{d}.csv")).values,
        "V": pd.read_csv(os.path.join(data_dir, f"V_d{d}.csv")).values,
    }
    for d in dims
}


numpy2ri.activate()
pandas2ri.activate()
np.random.seed(42)

# ─── FIGURE SETUP ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(10, 4), constrained_layout=True)
axes = axes.flatten()

for ax, (tag, dat) in zip(axes, datasets.items()):
    X, V = dat["X"], dat["V"]
    proj = X + V                       # veloviz needs a "projected" matrix
    cell_names = [f"cell_{i}" for i in range(X.shape[0])]

    # ── SEND MATRICES TO R AND RUN VELOVIZ ────────────────────────────────
    globalenv["curr"]       = X.T
    globalenv["proj"]       = proj.T
    globalenv["cell_names"] = StrVector(cell_names)

    r('''
      suppressPackageStartupMessages(library(veloviz))

      colnames(curr) <- cell_names
      colnames(proj) <- cell_names

      vv <- buildVeloviz(
              curr = curr, proj = proj,
              normalize.depth = FALSE, use.ods.genes = FALSE,
              alpha = 1, pca = FALSE, center = FALSE, scale = FALSE,
              k = 5, similarity.threshold = 0.25,
              distance.weight = 1, distance.threshold = 0.5,
              weighted = FALSE, verbose = FALSE
           )

      veloviz_embedding <- vv$fdg_coords           # 2-D coordinates
      cell_names_used   <- rownames(vv$fdg_coords) # subset actually connected
    ''')

    emb  = np.array(r["veloviz_embedding"])
    keep = [int(s.split("_")[-1]) for s in r["cell_names_used"]]

    Xk, Vk = X[keep], V[keep]          # same subset used in embedding

    # ── BUILD ANNData & PROJECT VELOCITY ONTO VELOVIZ BASIS ───────────────
    adata = anndata.AnnData(Xk)
    adata.layers["position"] = Xk
    adata.layers["velocity"] = Vk
    adata.obsm["X_veloviz"]  = emb

    scv.pp.neighbors(adata, use_rep="X")   # needed for projection helper
    scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
    scv.tl.velocity_embedding(adata, basis="veloviz")
    
    emb = adata.obsm["X_veloviz"]
    V_emb = adata.obsm["velocity_veloviz"]
    
    # ── QUIVER PLOT (yellow dots, long shafts, slim heads) ────────────────
    ax.scatter(
        emb[:, 0], emb[:, 1],
        c="#FFD700", s=150, alpha=0.10, edgecolors="none"
    )
    ax.quiver(
        emb[:, 0], emb[:, 1],
        V_emb[:, 0], V_emb[:, 1],
        color="black", angles="xy", scale_units="xy",
        scale=0.5, width=0.004,
        headwidth=3, headlength=4, headaxislength=3
    )

    ax.set_title(tag)
    ax.set_aspect("equal")
    ax.axis("off")

plt.show()